# Projet 4 : critiques IMDB, thèmes et prédiction du sentiment

ISSEA, ISE3 Data Science et Marketing, cours de Web Mining.

Ce notebook régénère les résultats du projet. Il s'enrichit à chaque phase. Les fonctions sont dans `src/` et tous les chiffres publiés sont écrits dans `reports/metrics.json`.

## Phase 1 : import, documentation et exploration

**Source.** Fichier Kaggle *IMDB Dataset of 50K Movie Reviews* (Lakshmipathi N), issu du *Large Movie Review Dataset* de Maas et al. (2011), ACL, p. 142-150.

**Variables.** `review` : texte de la critique, en anglais. `sentiment` : `positive` ou `negative`.

**Licence.** Kaggle affiche « Other (specified in description) », mais la description ne précise aucune licence : **[à vérifier]**. Le fichier brut n'est donc pas redistribué.

In [ ]:
import sys
from pathlib import Path

# Le notebook se lance depuis notebooks/ ou depuis la racine du projet
RACINE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RACINE))

import pandas as pd
from src.data import (SHA256_BRUT, charger_brut, cle_normalisee, cle_sans_ponctuation,
                      dedoublonner, diagnostiquer_doublons, empreinte_sha256)
from src import exploration as ex

GRAINE = 42  # graine commune du projet
empreinte = empreinte_sha256()
assert empreinte == SHA256_BRUT, 'Le fichier brut a changé'
brut = charger_brut(verifier=False)
print('SHA256 conforme :', empreinte)
print('Dimensions :', brut.shape)
brut.head(3)

### 1. Taille, classes et valeurs manquantes

Des classes équilibrées signifient qu'un modèle qui répond toujours la même chose a environ 50 % d'exactitude.

In [ ]:
classes_brut = brut['sentiment'].value_counts().to_dict()
manquants = {c: int(brut[c].isna().sum()) for c in ['review', 'sentiment']}
textes_vides = int(brut['review'].str.strip().eq('').sum())
print('Classes :', classes_brut)
print('Manquants :', manquants, '| textes vides :', textes_vides)

### 2. Doublons

On compare trois clés de plus en plus strictes. Les doublons sont retirés **avant** toute séparation : sinon, une même critique pourrait se trouver à la fois dans l'entraînement et dans le test.

In [ ]:
doublons = {
    'exact': diagnostiquer_doublons(brut, brut['review']),
    'normalise': diagnostiquer_doublons(brut, brut['review'].map(cle_normalisee)),
    'sans_ponctuation': diagnostiquer_doublons(brut, brut['review'].map(cle_sans_ponctuation)),
}
corpus = dedoublonner(brut, mode='sans_ponctuation')
classes_corpus = corpus['sentiment'].value_counts().to_dict()
print('Corpus retenu :', len(corpus), classes_corpus)
pd.DataFrame(doublons).T

### 3. Longueur des critiques par classe

In [ ]:
longueurs = ex.mesurer_longueurs(corpus)
ex.figure_longueurs(corpus, longueurs)

### 4. Balises HTML et autre bruit, notes écrites dans le texte

Une note écrite peut révéler directement l'étiquette, car celle-ci a été construite à partir de la note donnée par l'auteur. Ce point sera surveillé lors de l'analyse des erreurs et de l'interprétation du modèle.

In [ ]:
bruit = ex.mesurer_bruit(corpus)
notes = ex.mesurer_notes(corpus)
print({k: v['critiques'] for k, v in bruit.items()})
print('Critiques avec note :', notes['critiques_avec_note'], '| accord :', notes['accord_pct'], '%')
ex.figure_bruit(bruit, notes)

In [ ]:
ex.figure_notes(notes)

### 5. Mots fréquents par classe

Calcul descriptif sur tout le corpus dédoublonné. Ces listes ne servent pas à construire des variables du classifieur.

In [ ]:
table_mots = ex.compter_mots(corpus)
frequents = ex.mots_frequents(table_mots)
print('Vocabulaire :', len(table_mots), 'mots distincts')
ex.figure_mots_frequents(frequents)

In [ ]:
associes = ex.mots_associes(table_mots)
ex.figure_mots_associes(associes)

### 6. Limites du corpus

- **Étiquette binaire** : la note d'origine n'est pas fournie dans le fichier Kaggle.
- **Avis neutres exclus** : Maas et al. gardent les notes ≤ 4 et ≥ 7. Le modèle n'a jamais vu d'avis tiède.
- **Identifiants de films absents** : le corpus d'origine sépare les films entre entraînement et test, mais le fichier Kaggle ne permet plus de le faire. Une séparation aléatoire peut mettre un même film des deux côtés, donc le score de test peut être un peu optimiste.
- **Doublons** : retirés ici, avant la séparation.
- **Licence** : [à vérifier], voir plus haut.

In [ ]:
donnees = {
    'source': {
        'fichier': 'data/raw/IMDB Dataset.csv', 'sha256': empreinte,
        'kaggle': 'https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews',
        'origine': 'https://ai.stanford.edu/~amaas/data/sentiment/',
        'article': 'https://aclanthology.org/P11-1015/',
        'licence_kaggle_affichee': 'Other (specified in description)',
        'licence': '[à vérifier]', 'date_verification': '2026-11-21',
    },
    'brut': {'lignes': len(brut), 'colonnes': ['review', 'sentiment'], 'classes': classes_brut,
             'manquants': manquants, 'textes_vides': textes_vides},
    'doublons': doublons,
    'corpus_retenu': {'mode': 'sans_ponctuation', 'lignes': len(corpus), 'classes': classes_corpus,
                      'part_positive_pct': round(100 * classes_corpus['positive'] / len(corpus), 2)},
    'longueurs': longueurs, 'bruit': bruit, 'notes_dans_le_texte': notes,
    'mots': {'vocabulaire_distinct': int(len(table_mots)),
             'occurrences': {c: int(table_mots[c].sum()) for c in ['negative', 'positive']},
             'frequents_pour_1000': frequents, 'associes_log2': associes},
}
ex.enregistrer_metrics('donnees', donnees)
print('metrics.json mis à jour, clé donnees')

## Phase 2 : nettoyage et normalisation

**Pourquoi deux pipelines ?** Les deux tâches n'ont pas besoin des mêmes mots.

- **Pipeline A (classification)** : « not » change le sens d'une phrase. On garde donc les négations et les notes du type « 8/10 ».
- **Pipeline B (topics)** : « the », « not » ou « movie » ne décrivent aucun thème. On les retire, on lemmatise et on relie les expressions comme « special effects ».

Les règles vivent dans `src/preprocessing.py`, importé par ce notebook et par l'application. Les choix du pipeline A sont faits sur l'entraînement seul (80/20 stratifié, graine 42). Le test n'est pas utilisé pour les évaluer.

In [ ]:
from src import preprocessing as pp
from src.preprocessing_eval import executer

# Lit les caches de data/interim ; executer(recalculer=True) refait tout
bilan_pretraitement = executer()
textes = pd.read_parquet(RACINE / 'data' / 'interim' / 'textes_pretraites.parquet')
print(textes['partition'].value_counts().to_dict())

### 1. Exemple avant/après sur une vraie critique d'entraînement

Le pipeline A garde « did not ». Le pipeline B ne garde que les mots porteurs de thème, et relie « character development ».

In [ ]:
ligne = textes.loc[textes['id_brut'] == 1363].iloc[0]
assert ligne['partition'] == 'entrainement'
exemple = corpus.loc[corpus['id_brut'] == 1363, 'review'].iloc[0]
etapes = {
    'Texte brut': exemple,
    'Tronc commun': pp.tronc_commun(exemple),
    'Pipeline A': pp.pipeline_a(exemple),
    'Pipeline B': pp.pipeline_b(exemple),
}
for nom, valeur in etapes.items():
    print(f'{nom} :\n  {valeur}\n')
assert 'did not' in etapes['Pipeline A']
assert ligne['texte_a'] == etapes['Pipeline A'] and ligne['texte_b'] == etapes['Pipeline B']

### 2. Nettoyage : ce qui a changé

Un motif large comme `<[^>]+>` effacerait du vrai texte entre un `<` et un `>`. Le motif strict ne retire que des balises HTML reconnues.

In [ ]:
nettoyage = bilan_pretraitement['nettoyage']
negations = bilan_pretraitement['negations_entrainement']
pd.Series({
    'Critiques avec <br> avant': nettoyage['critiques_avec_br_avant'],
    'Critiques avec <br> après': nettoyage['critiques_avec_br_apres'],
    'Critiques où un motif large perdrait du texte': nettoyage['motif_large_critiques_avec_texte_perdu'],
    'Caractères que ce motif large perdrait': nettoyage['motif_large_caracteres_perdus'],
    'Critiques avec caractères Windows-1252': nettoyage['critiques_avec_caracteres_windows_1252'],
    'Négations sans apostrophe développées (dont, cant...)': nettoyage['negations_sans_apostrophe_developpees'],
    'Formes « cannot » développées': nettoyage['formes_cannot_developpees'],
    '« not » dans le brut (entraînement)': negations['not_dans_le_brut'],
    'Formes en n\'t dans le brut (entraînement)': negations['formes_nt_dans_le_brut'],
    '« not » après le pipeline A (entraînement)': negations['not_apres_pipeline_a'],
}, name='Valeur').to_frame()

### 3. Lemmatisation ou racinisation ?

- **Racinisation (Porter)** : coupe les suffixes par règles. Elle réduit plus le vocabulaire, mais produit des formes comme « movi » ou « stori ».
- **Lemmatisation (WordNet avec catégorie grammaticale)** : revient au mot du dictionnaire, comme « movie » ou « story ». Elle est plus lente.

La part de « vrais mots » vaut 100 % pour la lemmatisation par construction, car WordNet renvoie ses propres entrées. Pour la racinisation, c'est un maximum : une forme comme « wa » existe dans WordNet sans être le mot voulu.

In [ ]:
comparaison = bilan_pretraitement['lemmatisation_contre_racinisation']
chrono = bilan_pretraitement['chronometrage']
resume = pd.DataFrame({
    'Lemmatisation': {
        'Vocabulaire (mots distincts)': comparaison['lemmatisation']['vocabulaire'],
        'Réduction du vocabulaire (%)': comparaison['lemmatisation']['reduction_vocabulaire_pct'],
        'Occurrences modifiées restant de vrais mots (%)': comparaison['lemmatisation']['occurrences_modifiees_vrais_mots_pct'],
        'Temps sur 2 000 critiques (s)': chrono['lemmatisation_wordnet_pos']['secondes_sur_2000'],
    },
    'Racinisation': {
        'Vocabulaire (mots distincts)': comparaison['racinisation']['vocabulaire'],
        'Réduction du vocabulaire (%)': comparaison['racinisation']['reduction_vocabulaire_pct'],
        'Occurrences modifiées restant de vrais mots (%)': comparaison['racinisation']['occurrences_modifiees_vrais_mots_pct'],
        'Temps sur 2 000 critiques (s)': chrono['racinisation_porter']['secondes_sur_2000'],
    },
})
print('Vocabulaire d\'origine :', comparaison['vocabulaire_origine'])
display(pd.DataFrame(comparaison['exemples']).rename(columns={'mot': 'Mot', 'lemme': 'Lemme', 'racine': 'Racine'}))
resume

### 4. Validation croisée du pipeline A (5 plis, entraînement seul)

Le modèle est fixé (TF-IDF en mots et paires de mots, régression logistique) pour ne comparer que le prétraitement. Une option n'est retenue que si son gain moyen atteint 0,05 point **et** dépasse deux erreurs-types de l'écart entre plis. Cette règle a été fixée avant de lire les résultats.

In [ ]:
cv = bilan_pretraitement['validation_croisee_a']
tableau = pd.DataFrame(cv['resultats']).T[['exactitude', 'f1', 'roc_auc']].astype(float)
decisions = pd.DataFrame(cv['decisions']).T
tableau['gain (points)'] = decisions['gain_exactitude_points']
tableau['erreur-type (points)'] = decisions['erreur_type_points']
tableau['plis gagnants / 5'] = decisions['plis_gagnants']
tableau['retenue'] = decisions['ameliore']
effet = cv['effet_garder_negations']
print(f"Garder not, no, nor, never : {effet['gain_exactitude_points']:+.3f} point "
      f"(erreur-type {effet['erreur_type_points']:.3f}, {effet['plis_gagnants']} plis gagnants sur 5)")
print('Réglages retenus (vide = référence conservée) :', cv['reglages_retenus'])
tableau

**Lecture.** Aucune option ne dépasse le bruit entre plis : le pipeline A reste sans lemmatisation, sans racinisation et sans jeton de ponctuation.

Pour isoler le rôle de « not », on compare deux retraits de mots vides : l'un retire aussi « not, no, nor, never », l'autre les garde. Le seul fait de garder ces quatre mots améliore l'exactitude sur les 5 plis (chiffre ci-dessus). Le notebook de sentiment du cours retire ces mots : c'est un écart assumé.

Développer les contractions n'améliore pas le score. Explication probable, non testée séparément : la forme « didn » portait déjà la négation. On garde cette étape pour obtenir un seul jeton « not » et des coefficients lisibles.

### 5. Pipeline B : mots vides du domaine et expressions

Un mot présent dans au moins 20 % des critiques décrit peu un thème. On le retire, sauf s'il nomme un aspect du film (`story`, `character`, `plot`, `scene`, `actor`, `love`). Le filtre du cours (`no_above=0.5`) n'en retirerait que quatre. Les expressions sont choisies par information mutuelle normalisée (NPMI).

In [ ]:
b = bilan_pretraitement['pipeline_b']
domaine = b['mots_vides_domaine']
print('Mots au-dessus du seuil :', domaine['mots_au_dessus_du_seuil'],
      '| retirés :', domaine['mots_retires'], '| gardés :', domaine['mots_thematiques_gardes'])
print('Filtre du cours (> 50 %) :', domaine['mots_au_dessus_de_50_pct_regle_du_cours'])
print('Liste retenue :', ', '.join(b['liste_retenue']))
print('Expressions retenues :', b['bigrammes']['expressions_retenues'])
print('Stabilité corpus / entraînement (Jaccard) :', b['stabilite'])
print('Occurrences :', b['occurrences_avant_mots_domaine'], '->', b['occurrences_apres'],
      '| critiques vides :', b['critiques_vides_apres_pipeline_b'])
pd.DataFrame(b['bigrammes']['plus_frequentes']).head(12)

## Phase 3 : tokenisation et vectorisation

Un modèle ne lit pas des mots : il lit des nombres. La vectorisation transforme chaque critique en une ligne d'une grande matrice.

- **Sac de mots (BoW)** : chaque colonne compte les occurrences d'un terme. Il sert à LDA et à Naive Bayes.
- **TF-IDF** : le compte est pondéré, et un terme présent partout pèse moins. Il sert à NMF, à la régression logistique et au SVM.
- **Règle anti-fuite** : pour un modèle supervisé, le vocabulaire et les poids IDF s'apprennent sur l'entraînement seul, dans un `Pipeline`. Aucun score n'est calculé sur le test dans cette phase.

In [ ]:
from src import features as fx
from src import features_eval as fe

# Lit les caches de data/interim/phase3 ; fe.executer(recalculer=True) refait tout
bilan_vectorisation = fe.executer()
separation = bilan_vectorisation['separation']
cv_vecto = bilan_vectorisation['validation_croisee']

### 1. Séparation réutilisée et contrôlée

La séparation de `src/data.py` est recalculée, comparée au cache, puis sauvegardée dans `data/interim/separation_80_20.parquet`. On cherche aussi les textes communs aux deux parties, puis les **quasi-doublons** : des critiques recopiées avec quelques mots changés, que le dédoublonnage exact ne voit pas. Ce contrôle ne lit que les textes.

In [ ]:
print('Entraînement :', separation['entrainement'], '| test :', separation['test'],
      '| index communs :', separation['index_communs'])
print('Identique au cache :', separation['identique_aux_textes_en_cache'])
print('Textes communs :', separation['textes_communs'])
pd.DataFrame(separation['quasi_doublons_entre_parties']['seuils']).T

### 2. Taille du vocabulaire, creux, `min_df` et `max_df`

- **Matrice creuse** : presque toutes les cases valent zéro. On ne stocke que les valeurs non nulles.
- **`min_df`** retire les termes trop rares, souvent des fautes ou des noms isolés.
- **`max_df`** retire les termes trop fréquents. Il faut le manier avec prudence : « not » apparaît dans plus de 80 % des critiques.

In [ ]:
vocab = bilan_vectorisation['vocabulaire_pipeline_a']
print('Matrice de référence (1,2), min_df=5 :', vocab['matrice_reference']['profil_bow_et_tfidf'])
print('Effet de sublinear_tf sur le poids maximal moyen :', vocab['matrice_reference']['effet_sublinear_tf'])
fe.figure_vocabulaire(vocab['par_ngram'])

In [ ]:
pd.DataFrame(vocab['par_ngram']['1_2']['effet_min_df'])

In [ ]:
pd.DataFrame(vocab['par_ngram']['1_2']['effet_max_df'])

### 3. Effet de chaque réglage (validation croisée à 5 plis, entraînement seul)

On change **un seul réglage à la fois** autour de la référence de la phase 2. Un écart ne compte que s'il atteint 0,05 point et dépasse deux erreurs-types.

In [ ]:
reference_vecto = cv_vecto['resultats']['reference']
print('Référence :', reference_vecto['exactitude'], '| F1 :', reference_vecto['f1'], '| ROC-AUC :', reference_vecto['roc_auc'])
print('Pistes à régler en phase 5 :', cv_vecto['pistes_pour_la_phase_5'])
print('max_df=0,7 sans bigrammes :', cv_vecto['effet_max_df_0_7_sans_bigrammes'])
fe.figure_effet_reglages(cv_vecto['decisions_reglages'], reference_vecto)

In [ ]:
pd.DataFrame(cv_vecto['decisions_reglages']).T

**Lecture.** Les bigrammes et `sublinear_tf` aident nettement. Retirer « not » isolé ne coûte rien tant que les bigrammes comme « not good » restent, mais coûte sur les mots seuls. Plusieurs réglages sont testés sur les mêmes plis : un petit gain peut venir du hasard. Ils restent donc des **pistes pour la phase 5**, et la référence n'est pas modifiée ici.

### 4. BoW contre TF-IDF, là où ils servent

In [ ]:
comparaison_vecto = cv_vecto['bow_contre_tfidf']
tableau_vecto = pd.DataFrame({
    m: {'BoW': d['bow']['exactitude'], 'TF-IDF': d['tfidf']['exactitude'],
        'écart (points)': d['tfidf_moins_bow']['gain_exactitude_points'],
        'erreur-type': d['tfidf_moins_bow']['erreur_type_points'],
        'plis gagnés par TF-IDF': d['tfidf_moins_bow']['plis_gagnants']}
    for m, d in comparaison_vecto.items()
}).T
fe.figure_bow_tfidf(comparaison_vecto)
tableau_vecto

**Lecture.** TF-IDF l'emporte nettement pour le SVM et pour Naive Bayes. Pour la régression logistique, l'écart reste dans le bruit. Le cahier associe BoW à Naive Bayes : cette association reste la base, mais la phase 5 devra aussi tester TF-IDF pour ce modèle. Ici, TF-IDF inclut `sublinear_tf` : l'écart mesure donc la pondération complète, pas l'IDF seul.

### 5. La fuite par le vectoriseur, mesurée sur l'entraînement

Si le vectoriseur est ajusté **avant** la validation croisée, il connaît déjà les mots des plis de validation. On compare les deux protocoles sans toucher au test.

In [ ]:
bilan_vectorisation['fuite_vectoriseur_illustration']

### 6. Vectoriseurs des topics (usage descriptif)

LDA et NMF partagent le même vocabulaire, calculé sur tout le corpus dédoublonné **sans étiquette**. Ces matrices servent à décrire les thèmes. Elles ne doivent pas alimenter un classifieur évalué sur le test.

In [ ]:
topics_vecto = bilan_vectorisation['topics_pipeline_b']
print('Règle :', topics_vecto['regle_min_df'])
print('Réglages retenus :', topics_vecto['reglages_retenus'])
print('Matrice LDA :', topics_vecto['matrice_lda_bow'])
print('Termes les plus fréquents :', topics_vecto['frequence_documentaire_maximale'])
pd.DataFrame(topics_vecto['effet_min_df'])

## Phase 4 : thèmes des critiques et comparaison par polarité

**Question.** De quoi parlent les critiques, et quels thèmes vont avec les avis positifs ou négatifs ?

- **LDA sur BoW** (méthode du cours) : chaque critique est un mélange de thèmes, chaque thème une loi de probabilité sur les mots. Le modèle raisonne sur des comptages, d'où le sac de mots.
- **NMF sur TF-IDF** : la matrice documents-termes est factorisée en deux matrices positives. TF-IDF réduit le poids des mots présents partout.

**Règle anti-fuite.** Les topics sont ajustés sans étiquette sur les 49 575 critiques. La comparaison par polarité lit ensuite les étiquettes de tout le corpus, test compris. **Rien de cette phase n'alimente ni n'oriente le classifieur de la phase 5.**

In [ ]:
import json

from IPython.display import Image, display
from src import topics_eval as te

# Relit les caches de data/interim/phase4. Sans cache : python -m src.topics_grille, puis ce module.
bilan_topics = te.executer()
print(bilan_topics['usage'])

### 1. Choix du nombre de topics

Le cours fixe 5 thèmes sans optimisation : k = 5 reste dans la grille comme repère. La règle a été fixée **avant** de lire les résultats : diversité d'au moins 0,70, puis plus petit k à moins de 0,01 de la meilleure cohérence c_v, puis relecture humaine.

- **Cohérence** : les mots d'un topic apparaissent-ils ensemble dans les critiques ?
- **Diversité** : les topics utilisent-ils des mots différents ?

In [ ]:
print(bilan_topics['regle_choix_k'])
display(pd.DataFrame(bilan_topics['grille']).drop(columns=['min_df']).round(4))
pd.DataFrame(bilan_topics['choix_k']).T[['k', 'c_v', 'c_npmi', 'diversite', 'k_meilleure_npmi']]

In [ ]:
display(Image(filename=str(ex.DOSSIER_FIGURES / '09_choix_nombre_topics.png'), width=1000))

**Lecture.** Le compromis est visible : k = 5 mélange des sujets, surtout pour LDA, et un grand k ajoute des thèmes plus difficiles à présenter. Pour NMF, c_v préfère k = 8 alors que NPMI monte légèrement jusqu'à k = 20 : les deux mesures ne désignent donc pas le même k, et nous gardons le plus lisible.

### 2. Effet de `min_df` sur la qualité des topics

La phase 3 avait fixé `min_df = 10` sur un critère de couverture. On réajuste ici les modèles retenus avec 5, 10 et 20, à k constant.

In [ ]:
pd.DataFrame(bilan_topics['effet_min_df'])[['famille', 'k', 'min_df', 'termes_vocabulaire', 'c_v', 'c_npmi', 'diversite']].round(4)

**Lecture.** Pour NMF, `min_df` ne change presque rien. Pour LDA, les écarts n'ont pas de tendance et restent dans sa variation entre ajustements (section 3). `min_df = 10` est conservé : il garde un vocabulaire plus petit sans perte mesurable.

### 3. Stabilité : retrouve-t-on les mêmes topics ?

Chaque modèle est réajusté deux fois sur 80 % des critiques, avec une autre graine. Les topics sont appariés, puis comparés par l'indice de Jaccard de leurs 10 mots (1 = mêmes mots).

In [ ]:
pd.DataFrame({f: {c: v[c] for c in ('k', 'jaccard_moyen', 'jaccard_min', 'c_v_reajustements', 'npmi_reajustements')}
              for f, v in bilan_topics['stabilite'].items()}).T

**Lecture.** NMF retrouve l'essentiel de ses thèmes, LDA non. Le pic de LDA à k = 12 est du même ordre que la variation de sa cohérence entre ajustements : ce k est donc fragile. **NMF à 8 topics devient le modèle commun** de la comparaison, et LDA reste un contrôle.

### 4. Les topics retenus : mots, nom et extraits

Le modèle donne des mots et des poids. Les noms sont attribués par nous après relecture. La colonne `nature` distingue un **sujet** (de quoi parle la critique) d'un **jugement** (vocabulaire d'évaluation).

In [ ]:
print('Modèle commun :', bilan_topics['modele_commun'])
fiches = pd.DataFrame(bilan_topics['fiches_topics']['nmf'])
fiches['mots'] = fiches['mots'].str.join(', ')
pd.set_option('display.max_colwidth', None)
fiches[['topic', 'nom', 'nature', 'part_moyenne_pct', 'c_v', 'mots']].round(3)

In [ ]:
# Extraits relus : 2 à 3 critiques où le topic pèse le plus (choisies sans regarder les étiquettes)
extraits_topics = json.loads(te.CHEMIN_EXTRAITS.read_text(encoding='utf-8'))
for fiche in bilan_topics['fiches_topics']['nmf']:
    print(f"\n{fiche['topic']}. {fiche['nom']} ({fiche['nature']}) : {fiche['lecture']}")
    for e in extraits_topics['nmf'][fiche['topic']]:
        print(f"   [critique {e['id_brut']}, part {e['part_topic']:.2f}] {e['extrait']}")

In [ ]:
# Contrôle : les 12 topics de LDA
fiches_lda = pd.DataFrame(bilan_topics['fiches_topics']['lda'])
fiches_lda['mots'] = fiches_lda['mots'].str.join(', ')
fiches_lda[['topic', 'nom', 'nature', 'part_moyenne_pct', 'c_v', 'mots']].round(3)

### 5. Thèmes et polarité : ampleur des écarts

Un seul modèle est ajusté sur les deux classes. On compare ensuite la part moyenne de chaque topic dans les avis positifs et négatifs, avec un intervalle de confiance bootstrap à 95 % (2 000 tirages) et deux tailles d'effet :

- **d de Cohen** : l'écart de moyennes, en écarts-types ;
- **probabilité de supériorité** : la chance qu'une critique positive prise au hasard donne plus de place au topic qu'une critique négative (0,5 = aucune différence).

In [ ]:
colonnes = ['nom', 'nature', 'prevalence_positif_pct', 'prevalence_negatif_pct', 'ecart_points',
            'ic95_bas', 'ic95_haut', 'd_cohen', 'prob_superiorite', 'ampleur']
comparaison_nmf = bilan_topics['comparaison_polarite']['nmf']
display(pd.DataFrame(comparaison_nmf['topics'])[colonnes].round(3))
display(Image(filename=str(ex.DOSSIER_FIGURES / '10_topics_barres_divergentes.png'), width=1000))

In [ ]:
# Pourquoi la p-valeur ne suffit pas avec près de 50 000 critiques
print('Topics avec p < 0,001 :', comparaison_nmf['topics_p_inferieure_0_001'], 'sur', comparaison_nmf['nombre_topics'])
print('Topics avec un effet au moins faible (|d| >= 0,2) :', comparaison_nmf['topics_effet_au_moins_faible'])
print('Plus petit |d| significatif à 5 % avec ces effectifs (déduit) :', round(comparaison_nmf['d_minimal_significatif_5pct_deduit'], 4))
pd.DataFrame(comparaison_nmf['topics'])[['nom', 'ecart_points', 'd_cohen', 'prob_superiorite', 'p_valeur_welch']].round(4)

**Lecture.** Presque tous les écarts sont « significatifs », même ceux sans importance pratique. Le plus grand écart concerne le topic de **jugement** « Déception, film raté » : ce lien est en partie mécanique, car ses mots expriment déjà un avis. Parmi les **sujets**, les écarts sont nettement plus petits : personnages et récits de vie sont un peu plus présents dans les avis positifs, l'horreur un peu plus dans les négatifs.

Ce sont des **associations** : parler d'horreur ne rend pas un avis négatif. Les films d'horreur du corpus peuvent par exemple être de qualité différente des autres.

### 6. Mots qui distinguent les classes (log-odds avec a priori de Dirichlet)

Méthode de Monroe, Colaresi et Quinn (2008), équations 16, 19, 22 et 23 vérifiées dans l'article. L'a priori rapproche de zéro les écarts des mots rares, et le score z tient compte de l'incertitude de chaque mot.

In [ ]:
mots_d = bilan_topics['mots_distinctifs']
print(mots_d['reference'])
print('Sensibilité à alpha0 (mots communs sur 20) :', mots_d['sensibilite_alpha0'])
display(Image(filename=str(ex.DOSSIER_FIGURES / '11_mots_distinctifs_log_odds.png'), width=1000))
pd.DataFrame({'positif': [m['terme'] for m in mots_d['plus_positifs']],
              'z_positif': [round(m['z'], 1) for m in mots_d['plus_positifs']],
              'négatif': [m['terme'] for m in mots_d['plus_negatifs']],
              'z_négatif': [round(m['z'], 1) for m in mots_d['plus_negatifs']]})

### 7. Ce que l'on peut conclure

- Les avis se séparent surtout par leur **vocabulaire de jugement**, moins par leur sujet.
- Parmi les sujets, les associations existent mais restent **faibles ou négligeables**.
- NMF à 8 topics est le modèle retenu : plus cohérent, plus divers et bien plus stable que LDA sur ce corpus.

**Limites.** Les noms des topics sont une interprétation humaine. Les parts NMF sont des poids ramenés à 100 %, pas de vraies probabilités. Le corpus n'indique ni le film ni le genre : un écart peut venir des films critiqués plutôt que du thème lui-même. Rien ici n'alimente le classifieur de la phase 5.

## Phase 5 : analyse des sentiments, évaluation et gel

**Question.** Peut-on prédire le sentiment d'une critique, et quelle probabilité peut-on montrer à un utilisateur ?

**Protocole.** Tous les choix (réglages, notes écrites, calibration, zone d'incertitude) sont faits en validation croisée sur l'entraînement. Ils sont écrits dans `reports/phase05_choix_figes.json` et validés par un commit. Les prédictions du test sont ensuite calculées sans ses étiquettes et scellées par une empreinte. Les étiquettes du test ne sont lues qu'une fois.

Cette section **relit** les résultats gelés : elle ne réentraîne rien et ne rouvre pas le test. Les calculs sont dans `src/sentiment_grille.py`, `src/sentiment_eval.py` et `src/sentiment_test.py`.

In [ ]:
# Lecture des résultats gelés : aucun calcul sur le test ici
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from src import exploration as ex
from src import models as modeles_sentiment

sentiment = json.loads(ex.CHEMIN_METRICS.read_text(encoding='utf-8'))['sentiment']
print('Gel :', sentiment['gel'], '| commit des choix :', sentiment['commit_des_choix_figes'][:7])
print('1. Choix validés par commit :', sentiment['date_commit_des_choix'])
print('2. Prédictions du test      :', sentiment['date_predictions_test'])
print('3. Lecture des étiquettes   :', sentiment['date_lecture_etiquettes_test'])
print('Nombre de lectures des étiquettes du test :', sentiment['nombre_de_lectures_des_etiquettes_test'])

### 1. Référence lexicale : VADER

VADER additionne les scores d'un dictionnaire de mots, avec des règles pour la négation, les majuscules et la ponctuation. Il n'apprend rien sur nos données. C'est le point de comparaison : un modèle supervisé doit faire nettement mieux pour justifier sa complexité.

In [ ]:
v = sentiment['reference_vader_entrainement']
pd.Series({k: round(100 * v[k], 2) for k in ('exactitude', 'precision', 'rappel', 'f1', 'roc_auc')}, name='VADER, entraînement (%)').to_frame()

### 2. Réglage des hyperparamètres

Recherche en validation croisée à 5 plis sur l'entraînement : `min_df`, `max_df`, Naive Bayes sur BoW et sur TF-IDF, et le paramètre de régularisation de chaque modèle. La règle de choix est écrite avant les résultats : meilleure exactitude, puis, à moins de 0,05 point, le vocabulaire le plus petit et le modèle le plus régularisé.

**Problème rencontré.** La première grille plaçait le meilleur `C` sur sa borne haute, et `min_df = 5` battait 10 quand `C` était élevé. La grille a donc été étendue, toujours sur l'entraînement.

In [ ]:
r = sentiment['recherche_hyperparametres']
print(r['configurations_evaluees'], 'réglages,', r['ajustements'], 'ajustements, alertes de convergence :', r['alertes_convergence_totales'])
pd.DataFrame([
    {'famille': modeles_sentiment.LIBELLES_FAMILLES[k], 'paramètre': c['parametre'], 'valeur': c['valeur'], 'min_df': c['min_df'],
     'max_df': c['max_df'], 'exactitude CV (%)': round(100 * c['exactitude_cv'], 3), 'ROC-AUC CV': round(c['roc_auc_cv'], 4), 'termes': c['termes_moyens']}
    for k, c in r['choix'].items()
])

### 3. Les notes écrites dans le texte (« 8/10 »)

L'étiquette vient de la note donnée par l'auteur. Une note recopiée dans le texte peut donc souffler la réponse au modèle. On compare les modèles avec et sans ces motifs, **sur la validation**, avant d'ouvrir le test.

In [ ]:
n = sentiment['notes_ecrites']
print('Critiques concernées :', n['critiques_modifiees_par_le_retrait'], f"({n['part_de_l_entrainement_pct']} % de l'entraînement)")
print('Décision : retirer les notes =', n['retirer_les_notes'])
pd.DataFrame([
    {'famille': modeles_sentiment.LIBELLES_FAMILLES[k], 'avec notes (%)': round(100 * x['exactitude_avec_notes'], 3),
     'sans notes (%)': round(100 * x['exactitude_sans_notes'], 3), 'gain des notes (points)': x['effet_de_garder_les_notes']['gain_exactitude_points'],
     'erreur-type': x['effet_de_garder_les_notes']['erreur_type_points'], 'plis gagnants': x['effet_de_garder_les_notes']['plis_gagnants']}
    for k, x in n['familles'].items()
])

**Lecture.** Garder les notes améliore le score sur les 5 plis. Ce gain vient d'une information proche de l'étiquette : les notes sont donc **retirées**, avant la calibration et dans les modèles finaux. Le score publié est un peu plus bas, mais il mesure mieux la lecture du texte.

### 4. Calibration : une probabilité fiable

Quand un modèle annonce 80 %, environ 80 % des critiques concernées devraient être positives. Le **score de Brier** mesure l'écart entre probabilité et réalité (plus bas = mieux). La comparaison utilise une validation croisée imbriquée : le calibrateur n'a jamais vu le pli sur lequel il est mesuré.

In [ ]:
c = sentiment['calibration']
tableau = pd.DataFrame([
    {'famille': modeles_sentiment.LIBELLES_FAMILLES[x['famille']], 'méthode': x['methode'], 'Brier': round(x['brier_moyen'], 5),
     'log loss': round(x['log_loss_moyen'], 4), 'ECE': round(x['ece_moyen'], 4), 'exactitude (%)': round(100 * x['exactitude_moyen'], 3)}
    for x in c['candidats'].values()
])
print('Modèle retenu avant le test :', c['famille_retenue'], '+', c['methode_retenue'])
display(tableau.sort_values('Brier'))
display(Image(filename=str(ex.DOSSIER_FIGURES / '13_fiabilite_calibration.png'), width=1000))

### 5. Zone d'incertitude

Certaines critiques sont trop ambiguës pour un verdict. Les seuils ne sont pas choisis au hasard : on prend la **plus petite zone autour de 0,5** telle que l'exactitude atteigne 95 % en dehors, sur les probabilités de validation.

In [ ]:
z = sentiment['choix_figes']['zone_mesuree_hors_pli']
print(f"Seuils : {z['seuil_bas']} et {z['seuil_haut']}")
print(f"Validation : {100 * z['part_dans_la_zone']:.1f} % dans la zone, exactitude {100 * z['exactitude_dans_la_zone']:.1f} % dedans, {100 * z['exactitude_hors_zone']:.1f} % dehors")
display(pd.DataFrame(sentiment['choix_figes']['zone_stabilite_par_pli']).round(4).rename_axis('pli'))
display(Image(filename=str(ex.DOSSIER_FIGURES / '14_zone_incertitude.png'), width=1000))

### 6. Évaluation unique sur le test

Tous les modèles sont mesurés une seule fois, avec les choix déjà figés. Les intervalles à 95 % viennent d'un bootstrap de 2 000 tirages.

In [ ]:
lignes = []
for k, x in sorted(sentiment['test']['modeles'].items(), key=lambda p: -p[1]['exactitude']):
    i = x['ic95']['exactitude']
    lignes.append({'modèle': modeles_sentiment.LIBELLES_FAMILLES[k], 'exactitude (%)': round(100 * x['exactitude'], 2),
                   'IC 95 %': f"{100 * i['bas']:.2f} ; {100 * i['haut']:.2f}", 'précision (%)': round(100 * x['precision'], 2),
                   'rappel (%)': round(100 * x['rappel'], 2), 'F1 (%)': round(100 * x['f1'], 2), 'ROC-AUC': round(x['roc_auc'], 4),
                   'Brier': round(x['brier'], 5) if 'brier' in x else None})
display(pd.DataFrame(lignes))
display(Image(filename=str(ex.DOSSIER_FIGURES / '12_comparaison_modeles.png'), width=1000))
display(Image(filename=str(ex.DOSSIER_FIGURES / '15_matrice_confusion.png'), width=620))

In [ ]:
# Zone d'incertitude et quasi-doublons, mesurés sur les prédictions déjà scellées
t = sentiment['test']
z = t['zone_incertitude']
print(f"Zone sur le test : {100 * z['part_dans_la_zone']:.1f} % des critiques ({z['effectif_dans_la_zone']}), exactitude {100 * z['exactitude_dans_la_zone']:.1f} % dedans, {100 * z['exactitude_hors_zone']:.1f} % dehors")
q = t['quasi_doublons']
print(f"Avec les quasi-doublons : {100 * q['exactitude_avec']:.3f} %")
print(f"Sans les {q['critiques_retirees']} quasi-doublons : {100 * q['exactitude_sans']:.3f} % (écart {q['ecart_points']:.3f} point)")

### 7. Les mots qui pèsent le plus

Les coefficients de la régression logistique montrent les termes associés à chaque classe. Ils décrivent ce que le modèle a appris, pas une cause.

In [ ]:
co = sentiment['coefficients']
print('Termes communs avec le SVM parmi les 20 premiers :', co['termes_communs_avec_le_svm_sur_20'])
display(Image(filename=str(ex.DOSSIER_FIGURES / '16_coefficients_regression_logistique.png'), width=1000))

### 8. Analyse de 20 erreurs

Vingt erreurs du modèle retenu sont tirées au sort (10 faux positifs et 10 faux négatifs), puis lues et rangées selon une cause principale. Ce classement est une interprétation humaine sur un petit échantillon : il illustre les types d'erreurs, il n'en mesure pas la fréquence. Les textes ne sont pas reproduits ici.

In [ ]:
e = sentiment['analyse_erreurs']
print('Erreurs du modèle sur le test :', e['erreurs_totales_du_modele_sur_le_test'], '| faux positifs :', e['faux_positifs'], '| faux négatifs :', e['faux_negatifs'])
print('Parmi les 20 erreurs lues, dans la zone incertaine :', e['erreurs_lues_dans_la_zone_incertaine'])
display(pd.Series(e['repartition_des_20'], name='critiques').to_frame())
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(e['lignes'])[['id_brut', 'classe_reelle', 'probabilite_positive', 'cause', 'explication']])

### 9. Modèle sauvegardé

Le modèle retenu est sauvegardé avec `joblib`, en fichier compressé. Il attend le texte préparé par `preparer_texte` : notes écrites retirées, puis pipeline A.

In [ ]:
chemin = ex.RACINE / 'models' / 'sentiment_svm_lineaire.joblib'
if chemin.exists():
    modele_retenu = modeles_sentiment.charger_modele(chemin)
    exemples = ['A wonderful film with a moving story.', 'A boring and badly acted movie.', 'Good actors, but the story goes nowhere.']
    for texte, p in zip(exemples, modele_retenu.predict_proba([modeles_sentiment.preparer_texte(x) for x in exemples])[:, 1]):
        print(f'{p:.3f}  {texte}')
else:
    print('Modèle non présent : il se régénère avec src.sentiment_eval (étape gel).')

### 10. Ce que l'on peut conclure

- Les modèles supervisés font nettement mieux que le lexique VADER.
- Le SVM linéaire calibré est retenu **avant le test**, car sa probabilité est la plus fiable en validation.
- La zone d'incertitude isole une partie des critiques, sur lesquelles le modèle se trompe bien plus souvent.
- Les erreurs lues viennent surtout d'avis mitigés, de comparaisons, de résumés d'intrigue, de négations et du second degré.

**Limites.** Le corpus n'indique pas le film : une même œuvre peut être des deux côtés de la séparation, et le score peut être un peu optimiste. Les avis neutres sont absents du corpus, donc la zone d'incertitude n'est pas validée sur de vrais avis neutres. Les hyperparamètres ont été réglés avant le retrait des notes.

## Phase 6 : au-delà du cours, BERTopic et extension par phrase

**Question.** Un modèle fondé sur des embeddings pré-entraînés donne-t-il de meilleurs thèmes que NMF et LDA ?

**Cadre.** Les résultats gelés de la phase 5 ne changent pas. Le calcul complet des embeddings dépasse 15 minutes sur CPU : BERTopic travaille sur **5 000 critiques d'entraînement**, tirées par classe et quartile de longueur. Le test n'est pas relu.

In [ ]:
# Relecture du cache par défaut. RECALCULER = True relance les calculs manquants (paquets de requirements-avance.txt).
RECALCULER = False
import json
from IPython.display import Image, display
if RECALCULER:
    from src import bertopic_bilan, bertopic_model as bm, bertopic_phrases as bp
    bm.telecharger_encodeur(); bm.calculer_embeddings_critiques(); bm.calculer_embeddings_phrases()
    bm.executer_comparaison(); bp.croiser(); bertopic_bilan.executer()
bert = json.loads((ex.RACINE / 'reports' / 'metrics.json').read_text(encoding='utf-8'))['bertopic']
t = bert['temps']
print(f"2 000 critiques : {t['chrono_2000_critiques_s']:.0f} s | corpus extrapolé : {t['extrapolation_corpus_min']:.0f} min | avec marge de 2,4 : {t['extrapolation_corpus_avec_marge_2_4_min']:.0f} min")
print(f"5 000 critiques mesurées : {t['embeddings_5000_critiques_mesure_s'] / 60:.1f} min")
tr = bert['troncature']
print(f"Critiques au-delà de {tr['capacite_jetons_par_bloc']} jetons : {100 * tr['part_critiques_depassant_la_capacite']:.1f} % | jetons perdus sans découpage : {100 * tr['part_jetons_perdus_sans_decoupage']:.1f} %")

### 1. Embeddings et troncature

Un **embedding** représente un texte par un vecteur : deux textes de sens proche ont des vecteurs proches. Le modèle lit au plus 256 jetons. Les critiques longues sont donc **découpées en blocs de phrases**, puis leurs vecteurs sont moyennés.

Les **mots vides restent** avant l'embedding, car « not good » et « good » n'ont pas le même sens. Ils sont retirés seulement pour afficher les mots des topics.

### 2. Comparaison sur les mêmes critères

Tous les modèles utilisent les mêmes mots possibles et le même calcul de cohérence. **Différence de prétraitement :** BERTopic regroupe les critiques sur leur texte presque complet, NMF et LDA sur des lemmes sans mots vides.

In [ ]:
lignes = []
noms = {'nmf_k8_corpus': 'NMF k=8 (corpus)', 'lda_k12_corpus': 'LDA k=12 (corpus)', 'nmf_k8_echantillon': 'NMF k=8 (5 000)',
        'lda_k12_echantillon': 'LDA k=12 (5 000)', 'bertopic_natif': 'BERTopic natif', 'bertopic_k8': 'BERTopic k=8', 'bertopic_k12': 'BERTopic k=12'}
for cle, nom in noms.items():
    x = bert['comparaison'][cle]
    lignes.append({'modèle': nom, 'topics': x['n_topics'], 'c_v': round(x['c_v'], 3), 'NPMI': round(x['c_npmi'], 3),
                   'diversité': round(x['diversite'], 3), 'hors topic %': round(100 * x['part_hors_topic'], 1),
                   'stabilité': None if x['stabilite_jaccard'] is None else round(x['stabilite_jaccard'], 3)})
display(pd.DataFrame(lignes))
c = bert['concentration']['bertopic_k8']
print(f"BERTopic k=8 : plus grand topic = {100 * c['part_du_plus_grand_topic_parmi_les_classees']:.1f} % des critiques classées ; {c['topics_de_10_critiques_ou_moins']} topics de 10 critiques ou moins")
print('Poids :', bert['poids'])
print(bert['conclusion'])
display(Image(filename=str(ex.DOSSIER_FIGURES / '17_bertopic_comparaison.png')))

**Lecture.** BERTopic réduit à 8 topics a la meilleure c_v, mais ce score flatte des micro-topics d'une dizaine de critiques sur une seule œuvre. Plus de la moitié des critiques restent hors topic (**topic −1** : critiques que HDBSCAN ne rattache à aucun groupe dense), un topic fourre-tout domine les autres et les mots changent quand on réajuste. **NMF k=8 reste le modèle commun.**

**Limite.** La première règle de choix de HDBSCAN donnait un topic de 88 % des critiques. Elle a été révisée après avoir vu la grille, et la valeur retenue est au bord bas de la grille.

### 3. Topics par polarité (`topics_per_class`)

Le graphique compare la part des critiques de chaque topic entre avis positifs et négatifs. Il s'agit d'une association, mesurée sur l'échantillon d'entraînement.

In [ ]:
display(pd.DataFrame([{'topic': l['nom'], 'critiques': l['critiques'], 'écart (points)': round(l['ecart_points'], 2),
                       'IC95 bas': round(l['ic95_bas'], 2), 'IC95 haut': round(l['ic95_haut'], 2)} for l in bert['polarite_k8']['topics']]))
display(Image(filename=str(ex.DOSSIER_FIGURES / '18_bertopic_polarite.png')))

### 4. Extension exploratoire : thèmes des phrases et sentiment des phrases

**Attention.** Le corpus n'a pas d'étiquette par phrase et le classifieur est calibré sur des critiques entières. Les parts ci-dessous décrivent les **scores du modèle**, pas le sentiment réel. Aucune exactitude par phrase n'est publiée. Le seul contrôle possible est l'accord avec VADER, lui aussi imparfait.

Les scores viennent d'une copie de la recette gelée, réapprise sans les 5 000 critiques de l'échantillon. Le modèle gelé n'est pas modifié.

In [ ]:
x = bert['extension_phrases']
a = x['accord_vader']
print(f"{x['critiques']} critiques, {x['phrases_scorees']} phrases, {100 * x['part_phrases_hors_aspect']:.1f} % hors aspect")
print(f"Accord avec VADER (phrases non neutres) : {100 * a['accord_brut_vader_non_neutre']:.1f} %, kappa {a['kappa_cohen_vader_non_neutre']:.3f}, Spearman {a['correlation_spearman_toutes_phrases']:.3f}")
for nom, s in x['critiques_mixtes'].items():
    print(f"{nom} ({s['bas']} / {s['haut']}) : {100 * s['critiques_positives_avec_une_phrase_negative']:.1f} % des critiques positives ont une phrase à score négatif ; {100 * s['critiques_negatives_avec_une_phrase_positive']:.1f} % des négatives ont une phrase à score positif")
display(Image(filename=str(ex.DOSSIER_FIGURES / '19_phrases_aspects_sentiment.png')))

**Lecture prudente.** Les scores suggèrent que beaucoup de critiques mêlent les deux tonalités et que chaque aspect reçoit des phrases des deux signes. L'accord avec VADER reste modeste (kappa d'environ 0,29) : ces parts sont une piste de lecture, pas une mesure validée.